# Object detection with YOLO

**Thursday, 13:00.** First part of the afternoon block.

YOLO is **not** a generative model - it is a *discriminative* detector trained
for one task. That is exactly why it is here: to see the difference between a
narrow model that does one thing reliably, and a general model that you simply
*describe* the task to.

At the end we wrap YOLO into a function called `detect()`. **That function is
handed to the AI agent at 16:00**, so keep this notebook open.

Everything runs on **CPU**. No GPU is needed - do not request one.

In [ ]:
# --- SETUP: run this first ---  [lares-setup-v1]
# works in Colab and locally; safe to run more than once
import os, sys, subprocess
REPO = "ai_bootcamp_foundations"
if "google.colab" in sys.modules:
    if not os.path.isdir(f"/content/{REPO}"):
        subprocess.run(["git", "clone", "-q",
                        f"https://github.com/hrvojenovak/{REPO}.git"],
                       cwd="/content", check=True)
    os.chdir(f"/content/{REPO}/notebooks")
print("cwd:", os.getcwd(), "| data ok:", os.path.isdir("../data"))

## Part 0: Setup

### Install
Takes about 30 s. If Colab asks to **Restart runtime**, restart and run all
cells from the top - the SETUP cell is written to survive that.

In [ ]:
%pip install -q ultralytics

import os
os.environ["YOLO_VERBOSE"] = "False"

# ultralytics sends anonymous telemetry by default; turn it off
from ultralytics import settings
settings.update({"sync": False})

import ultralytics, torch
torch.set_num_threads(2)          # free Colab has 2 vCPUs
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__)
print("CUDA available:", torch.cuda.is_available(), " <- not needed here")

## Part 1: The model

`yolo11n` is the "nano" variant: **5.6 MB**, ~2.6M parameters. Larger ones
exist (`s`, `m`, `l`, `x`) - `yolo11s` is 19 MB and roughly 2x slower.

All of them know the **same 80 classes**. A bigger model recognises the same
things more accurately; it does not recognise *more* things. The class list
comes from the training dataset, not from network capacity.

In [ ]:
from ultralytics import YOLO

# weights download automatically on first use (~5.6 MB)
model = YOLO("yolo11n.pt")

n_params = sum(p.numel() for p in model.model.parameters())
print(f"parameters: {n_params/1e6:.1f}M")
print(f"classes known: {len(model.names)}")
print(f"examples: {[model.names[i] for i in [0, 2, 5, 15, 39, 63]]}")

## Part 2: First inference and timing

We measure on purpose. The point worth remembering: **inference and training
are completely different resource problems.** People conflate them and then
assume everything needs a GPU.

In [ ]:
import time, urllib.request

IMG = "bus.jpg"
if not os.path.exists(IMG):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/ultralytics/ultralytics"
        "/main/ultralytics/assets/bus.jpg", IMG)

model(IMG, verbose=False)                     # warm-up, the first call is slower

t = time.time()
results = model(IMG, verbose=False)
dt = (time.time() - t) * 1000

print(f"inference: {dt:.0f} ms on CPU\n")
for box in results[0].boxes:
    print(f"  {model.names[int(box.cls)]:12s} confidence {float(box.conf):.2f}")

## Part 3: Looking at the result

`results[0].plot()` draws the boxes. It returns a **BGR** array (OpenCV
convention), so the channels have to be reversed before display - a classic
trap.

In [ ]:
import matplotlib.pyplot as plt

annotated = results[0].plot()[:, :, ::-1]     # BGR -> RGB

plt.figure(figsize=(7, 9))
plt.imshow(annotated)
plt.axis("off")
plt.title(f"yolo11n - {len(results[0].boxes)} detections, {dt:.0f} ms (CPU)")
plt.show()

## Part 4: `detect()` - a tool for the agent

This is the most important cell in the notebook.

So far we have used YOLO interactively. Now we close it into a function with a
**clean signature**: `detect(path) -> list[dict]`. Input is a path, output is
plain Python/JSON.

Why: **to an agent, a tool is anything with a defined interface**, not just
text. When the agent gets `detect` alongside `run_python` at 16:00, it will be
able to "see" images - and we will not have written a single new line of model
code.

In [ ]:
def detect(path: str, conf: float = 0.25) -> list[dict]:
    """Detect objects in an image.

    Args:
        path: path to the image
        conf: minimum confidence (0-1)

    Returns:
        List of detections: label, confidence, box [x1, y1, x2, y2].
    """
    if not os.path.exists(path):
        return [{"error": f"no such file: {path}"}]

    r = model(path, conf=conf, verbose=False)[0]
    return [
        {
            "label": model.names[int(b.cls)],
            "confidence": round(float(b.conf), 3),
            "box": [round(v) for v in b.xyxy[0].tolist()],
        }
        for b in r.boxes
    ]


import json
print(json.dumps(detect(IMG), indent=1))

## Part 5: Exercise - where YOLO stops

Run the detector on images from **your own** domain: an overhead line, a PV
panel, a substation, an insulator.

Expect it to fail. YOLO knows **80 COCO classes** and that is all you get
without fine-tuning. A PV panel will come back as `tv`, or as nothing at all.

**That is not a bug, that is the point.** A narrow model does exactly what it
was trained for and nothing beyond it. Keep that in mind when, an hour from
now, you see a general model that you only have to *describe* the task to.

The helper below shows the annotated image as well, and says so explicitly
when nothing was found - otherwise you cannot tell "no detections" from "the
cell did not run".

In [ ]:
import matplotlib.pyplot as plt

def show_detections(path: str, conf: float = 0.25) -> list[dict]:
    """Detect and display in one pass - one inference, two outputs."""
    if not os.path.exists(path):
        print(f"  no such file: {path}")
        return []
    try:
        r = model(path, conf=conf, verbose=False)[0]
    except Exception as e:
        print(f"  not an image, or unreadable: {e}")
        return []

    plt.figure(figsize=(8, 6))
    plt.imshow(r.plot()[:, :, ::-1])          # BGR -> RGB
    plt.axis("off")
    n = len(r.boxes)
    plt.title(f"{os.path.basename(path)} - {n} detections" if n
              else f"{os.path.basename(path)} - NOTHING DETECTED")
    plt.show()

    return [{"label": model.names[int(b.cls)],
             "confidence": round(float(b.conf), 3),
             "box": [round(v) for v in b.xyxy[0].tolist()]}
            for b in r.boxes]


# try images shipped in the repo first, then offer an upload
import glob
domain_imgs = sorted(glob.glob("../data/img_yolo/*"))
for path in domain_imgs:
    print(f"\n=== {os.path.basename(path)} ===")
    found = show_detections(path)
    print(json.dumps(found, indent=1) if found else "  NOTHING DETECTED")

if "google.colab" in sys.modules:
    print("\n--- or upload your own image ---")
    from google.colab import files
    uploaded = files.upload()
    for name in uploaded:
        print(f"\n=== {name} ===")
        found = show_detections(name)
        print(json.dumps(found, indent=1) if found else "  NOTHING DETECTED")
elif not domain_imgs:
    print("Locally: show_detections('path/to/image.jpg')")

## Part 6: Wider vocabularies

If 80 classes are not enough, there are three ways out.

**Open Images weights: 601 classes.** `yolov8n-oiv7.pt` adds things COCO lacks
(`Tree`, `Building`, `Window`, `Wheel`, `Tool`). More classes are not free
though - the same capacity is spread thinner, so per-class accuracy is lower.

**Open-vocabulary models: no fixed list.** YOLO-World and YOLOE take a text
prompt: `model.set_classes(["solar panel", "insulator"])`. This is the bridge
to the afternoon - language becomes the interface to a vision model. Note that
the first `set_classes()` call downloads a CLIP text encoder, which is a large
extra download.

**Fine-tuning on your own data.** The only route to reliable detection of
domain objects. The model is the easy part; **labelling images is the work.**

## Takeaways

**80 classes, and that is all.** Your domain needs fine-tuning on labelled
data - and labelling is a project, not a detail.

**Inference does not need a GPU.** Measured: ~100 ms per image on CPU. GPUs are
for *training* and for *scale*, not for learning.

**Model size changes accuracy, not vocabulary.** `yolo11x` has 20x more
parameters than `yolo11n` and exactly the same 80 classes.

**Licence: `ultralytics` is AGPL-3.0.** That is a copyleft that extends to
network use. Irrelevant for a course, **potentially a problem in your company.**
More permissive alternatives: torchvision detectors (BSD), YOLOX (Apache 2.0),
RT-DETR.

**`detect()` stays.** The agent receives it as a tool at 16:00.